In [ ]:
{
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# Medical Search Model Training\n",
                "Train and update search model based on user queries and feedback"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "import sys\n",
                "from pathlib import Path\n",
                "import os\n",
                "\n",
                "# Add project root to path\n",
                "project_root = Path.cwd().parent\n",
                "sys.path.append(str(project_root))\n",
                "\n",
                "from src.data.database import Database\n",
                "from sentence_transformers import SentenceTransformer\n",
                "import pandas as pd\n",
                "import numpy as np\n",
                "from datetime import datetime, timezone\n",
                "from sklearn.model_selection import train_test_split"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "# Load search patterns\n",
                "db = Database()\n",
                "\n",
                "query = \"\"\"\n",
                "SELECT \n",
                "    sl.query,\n",
                "    sl.processed_query,\n",
                "    sl.search_type,\n",
                "    sl.results_count,\n",
                "    qe.expanded_terms\n",
                "FROM search_logs sl\n",
                "LEFT JOIN query_expansion_logs qe ON sl.query = qe.original_query\n",
                "WHERE sl.results_count > 0\n",
                "\"\"\"\n",
                "\n",
                "search_data = pd.read_sql(query, db.engine)"
            ]
        },
        {
            "cell_type": "code",
            "source": [
                "# Train custom embeddings\n",
                "from sentence_transformers import SentenceTransformer, InputExample, losses\n",
                "from torch.utils.data import DataLoader\n",
                "\n",
                "model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')\n",
                "\n",
                "# Prepare training examples\n",
                "train_examples = []\n",
                "for _, row in search_data.iterrows():\n",
                "    train_examples.append(InputExample(\n",
                "        texts=[row['query'], row['processed_query']],\n",
                "        label=1.0\n",
                "    ))\n",
                "\n",
                "# Create data loader\n",
                "train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)\n",
                "\n",
                "# Use cosine similarity loss\n",
                "train_loss = losses.CosineSimilarityLoss(model)\n",
                "\n",
                "# Train the model\n",
                "model.fit(\n",
                "    train_objectives=[(train_dataloader, train_loss)],\n",
                "    epochs=3,\n",
                "    warmup_steps=100\n",
                ")"
            ]
        }
    ]
}